# Libri6Mix prep — auxiliary notebook (internet ON, CPU is fine)

Stages everything the offline training notebook needs into `/kaggle/working/`:

1. pip deps (installed with `--target`, importable via `sys.path`)
2. the SR_CorrNet_SS repo (cloned from GitHub)
3. the pretrained 2–5-speaker checkpoint from HuggingFace
4. **Libri{2..6}Mix**: N-speaker mixtures generated from LibriSpeech (2mix–6mix, train/valid/test)

**Before running:** attach a LibriSpeech dataset (Add Data) containing `train-clean-100`
and `dev-clean`, and point `LIBRI_TRAIN_ROOTS` / `LIBRI_EVAL_ROOTS` below at it.
**After running:** Save Version so the output persists, then attach this notebook's
output to the training notebook.


In [ ]:
# ---- parameters -------------------------------------------------------------
import glob

# Point these at the attached LibriSpeech dataset (dirs that contain speaker-id subdirs).
# Common layouts: /kaggle/input/<slug>/LibriSpeech/train-clean-100 or /kaggle/input/<slug>/train-clean-100
LIBRI_TRAIN_ROOTS = glob.glob("/kaggle/input/*/LibriSpeech/train-clean-100") or glob.glob("/kaggle/input/*/train-clean-100")
LIBRI_EVAL_ROOTS  = glob.glob("/kaggle/input/*/LibriSpeech/dev-clean") or glob.glob("/kaggle/input/*/dev-clean")

# mixtures per speaker count, order matches COUNTS. Weighted toward 6mix.
COUNTS  = "2,3,4,5,6"
N_TRAIN = "2000,2000,3000,5000,8000"   # ~7.5 GB of wavs
N_VALID = "500,500,750,1250,2000"      # fixed-seed validation set (~1.9 GB)
N_TEST  = "500,500,500,500,500"        # fixed-seed test set (~1.4 GB)
SEED    = 1234

print("train roots:", LIBRI_TRAIN_ROOTS)
print("eval roots: ", LIBRI_EVAL_ROOTS)
assert LIBRI_TRAIN_ROOTS and LIBRI_EVAL_ROOTS, "attach a LibriSpeech dataset and set the roots above"


In [ ]:
# ---- 1. pip deps into /kaggle/working/deps -----------------------------------
# Everything engine.py imports that the offline GPU image may not ship.
# matplotlib must be <3.10: the repo's util_writer uses the removed tostring_rgb().
!pip install --quiet --target=/kaggle/working/deps \
    loguru rotary-embedding-torch mir_eval tensorboardX torchinfo thop ptflops \
    "matplotlib<3.10" soundfile librosa pyyaml tqdm scipy huggingface_hub
print("deps staged")


In [ ]:
# ---- 2. repo + 3. pretrained checkpoint ---------------------------------------
!git clone --depth 1 https://github.com/dmlguq456/SR_CorrNet_SS.git /kaggle/working/SR_CorrNet_SS
from huggingface_hub import hf_hub_download
for f in ["model.pt", "config.yaml"]:
    hf_hub_download("shinuh/sr-corrnet-ss-1ch-wsj-var-2-5spk", f, local_dir="/kaggle/working/ckpt")
print("checkpoint staged")


In [ ]:
# ---- 4a. mixture generator (verified locally) ---------------------------------
gen_src = '"""Generate Libri{2..6}Mix: N-speaker mixtures from LibriSpeech for SR-CorrNet fine-tuning.\n\nLayout (matches the repo\'s SCP convention, scp_dir/<n>mix/{tr,cv,tt}_{mix,sK}.scp):\n    out_dir/<n>mix/{tr,cv,tt}/{mix,s1..sN}/<key>.wav      8 kHz, 16-bit PCM, 4 s\n    scp_dir/<n>mix/{tr,cv,tt}_mix.scp                      lines: "<key> <abs path>"\n    scp_dir/<n>mix/{tr,cv,tt}_s<K>.scp\n\nConstruction per mixture: N distinct speakers, one random utterance each, resampled\n16k->8k, random 4s crop (zero-padded if shorter), per-source gain U[-5,+5] dB,\nmixture = sum(sources); if peak > 0.9 the mixture AND all sources are rescaled by the\nsame factor (never independently — that would corrupt SI-SNR targets).\n"""\nimport argparse\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport librosa\nimport soundfile as sf\n\nSR = 8000\nMAX_LEN = 32000  # 4 s @ 8 kHz\n\n\ndef index_speakers(roots):\n    """{speaker_id: [flac paths]} across one or more LibriSpeech subset roots.\n\n    Only utterances >= 4 s are kept: a silent (all-zero) source segment gives the\n    SI-SNR loss a NaN gradient (torch.norm at 0), which destroys training.\n    """\n    spk = {}\n    for root in roots:\n        for f in Path(root).rglob("*.flac"):\n            info = sf.info(str(f))\n            if info.frames / info.samplerate >= MAX_LEN / SR:\n                spk.setdefault(f.parts[-3], []).append(f)\n    return {k: sorted(v) for k, v in spk.items() if len(v) >= 2}\n\n\ndef load_clip(path, rng):\n    wav, _ = librosa.load(path, sr=SR)\n    if len(wav) > MAX_LEN:\n        off = rng.integers(0, len(wav) - MAX_LEN)\n        wav = wav[off:off + MAX_LEN]\n    else:\n        wav = np.pad(wav, (0, MAX_LEN - len(wav)))\n    return wav\n\n\ndef make_partition(speakers, n_spks, n_mix, out_wav, out_scp, part, rng):\n    spk_ids = sorted(speakers)\n    dirs = ["mix"] + [f"s{k}" for k in range(1, n_spks + 1)]\n    for d in dirs:\n        (out_wav / d).mkdir(parents=True, exist_ok=True)\n    scp = {d: [] for d in dirs}\n    for i in range(n_mix):\n        chosen = rng.choice(spk_ids, size=n_spks, replace=False)\n        srcs = []\n        for sid in chosen:\n            utt = speakers[sid][rng.integers(len(speakers[sid]))]\n            gain = 10 ** (rng.uniform(-5, 5) / 20)\n            srcs.append(load_clip(utt, rng) * gain)\n        mixture = np.sum(srcs, axis=0)\n        peak = np.abs(mixture).max()\n        if peak > 0.9:\n            srcs = [s * (0.9 / peak) for s in srcs]\n            mixture = mixture * (0.9 / peak)\n        key = f"{part}_{n_spks}mix_{i:06d}"\n        for d, wav in zip(dirs, [mixture] + srcs):\n            p = out_wav / d / f"{key}.wav"\n            sf.write(p, wav.astype(np.float32), SR, subtype="PCM_16")\n            scp[d].append(f"{key} {p.resolve()}\\n")\n    prefix = {"tr": "tr", "cv": "cv", "tt": "tt"}[part]\n    out_scp.mkdir(parents=True, exist_ok=True)\n    for d in dirs:\n        name = f"{prefix}_{\'mix\' if d == \'mix\' else d}.scp"\n        with open(out_scp / name, "w") as f:\n            f.writelines(scp[d])\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--train_roots", nargs="+", required=True,\n                    help="LibriSpeech subset dirs for training speakers (e.g. train-clean-100)")\n    ap.add_argument("--eval_roots", nargs="+", required=True,\n                    help="LibriSpeech subset dirs for valid/test speakers (e.g. dev-clean)")\n    ap.add_argument("--out_wav", required=True)\n    ap.add_argument("--out_scp", required=True)\n    ap.add_argument("--counts", default="2,3,4,5,6")\n    ap.add_argument("--n_train", default="2000,2000,3000,5000,8000",\n                    help="mixtures per count (same order as --counts); weights training toward high N")\n    ap.add_argument("--n_valid", default="500,500,750,1250,2000")\n    ap.add_argument("--n_test", default="500,500,500,500,500")\n    ap.add_argument("--seed", type=int, default=1234)\n    args = ap.parse_args()\n\n    counts = [int(c) for c in args.counts.split(",")]\n    n_tr = [int(c) for c in args.n_train.split(",")]\n    n_cv = [int(c) for c in args.n_valid.split(",")]\n    n_tt = [int(c) for c in args.n_test.split(",")]\n\n    train_spk = index_speakers(args.train_roots)\n    eval_spk = index_speakers(args.eval_roots)\n    # valid and test must not share speakers: split the eval speaker set in half\n    eval_ids = sorted(eval_spk)\n    random.Random(args.seed).shuffle(eval_ids)\n    cv_spk = {k: eval_spk[k] for k in eval_ids[: len(eval_ids) // 2]}\n    tt_spk = {k: eval_spk[k] for k in eval_ids[len(eval_ids) // 2:]}\n    print(f"speakers: train={len(train_spk)} valid={len(cv_spk)} test={len(tt_spk)}")\n\n    for n, a, b, c in zip(counts, n_tr, n_cv, n_tt):\n        sub = f"{n}mix"\n        for part, spk, cnt in [("tr", train_spk, a), ("cv", cv_spk, b), ("tt", tt_spk, c)]:\n            # fixed seed per (count, partition): valid/test are identical across runs\n            rng = np.random.default_rng(args.seed + 1000 * n + {"tr": 0, "cv": 1, "tt": 2}[part])\n            make_partition(spk, n, cnt, Path(args.out_wav) / sub / part,\n                           Path(args.out_scp) / sub, part, rng)\n            print(f"{sub}/{part}: {cnt} mixtures done")\n\n\nif __name__ == "__main__":\n    main()\n'
with open('/kaggle/working/generate_libri_mix.py', 'w', encoding='utf-8') as f:
    f.write(gen_src)
print('generator written')


In [ ]:
# ---- 4b. generate Libri{2..6}Mix (takes a while on CPU: ~1-2 h) ---------------
import subprocess, sys
cmd = [sys.executable, "/kaggle/working/generate_libri_mix.py",
       "--train_roots", *LIBRI_TRAIN_ROOTS, "--eval_roots", *LIBRI_EVAL_ROOTS,
       "--out_wav", "/kaggle/working/libri_mix/wav", "--out_scp", "/kaggle/working/libri_mix/scp",
       "--counts", COUNTS, "--n_train", N_TRAIN, "--n_valid", N_VALID, "--n_test", N_TEST,
       "--seed", str(SEED)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
!du -sh /kaggle/working/libri_mix /kaggle/working/deps /kaggle/working/SR_CorrNet_SS /kaggle/working/ckpt


Done. **Save Version** (so `/kaggle/working` persists), then attach this notebook's output to the training notebook as input data.